# 🐍 Day 7: Mini-Project - API Aggregator

- Fetch data from multiple APIs
- Aggregate and normalize responses
- Flask endpoint to serve aggregated data
- Error handling and timeouts

## Project Overview

We'll build an API aggregator that:
- Fetches from httpbin.org (mock APIs)
- Combines JSON and IP info
- Exposes a single Flask endpoint `/api/dashboard`
- Handles errors gracefully

## Step 1: Fetch a Single API

Create a helper to fetch JSON from a URL with timeout and error handling.

In [ ]:
import requests

def fetch_json(url, timeout=5):
    """Fetch URL and return JSON dict or None on error."""
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except requests.RequestException:
        return None

data = fetch_json("https://httpbin.org/json")
print("Keys:", data.keys() if data else "Failed")

## Step 2: Fetch Multiple APIs

Fetch from several endpoints in parallel (or sequentially for simplicity).

In [ ]:
def fetch_all(urls):
    results = {}
    for i, url in enumerate(urls):
        data = fetch_json(url)
        results[f"source_{i}"] = data if data else {"error": "fetch_failed"}
    return results

urls = [
    "https://httpbin.org/json",
    "https://httpbin.org/get",
    "https://httpbin.org/ip"
]
aggregated = fetch_all(urls)
print("Sources:", list(aggregated.keys()))

## Step 3: Normalize the Response

Create a unified structure for the dashboard.

In [ ]:
def build_dashboard(aggregated):
    dashboard = {
        "sources": len(aggregated),
        "data": {},
        "errors": []
    }
    for key, val in aggregated.items():
        if val and "error" not in val:
            dashboard["data"][key] = val
        else:
            dashboard["errors"].append(key)
    return dashboard

dash = build_dashboard(aggregated)
print(dash)

## Step 4: Flask App with Dashboard Route

Create a route that fetches and returns the aggregated data.

In [ ]:
from flask import Flask, jsonify

app = Flask(__name__)

API_URLS = [
    "https://httpbin.org/json",
    "https://httpbin.org/ip",
    "https://httpbin.org/uuid"
]

@app.route("/api/dashboard")
def dashboard():
    aggregated = fetch_all(API_URLS)
    result = build_dashboard(aggregated)
    return jsonify(result)

print("Flask app with /api/dashboard defined.")

## Step 5: Add Caching (Optional)

Cache results for a few seconds to avoid hammering APIs.

In [ ]:
import time

cache = {"data": None, "timestamp": 0}
CACHE_TTL = 30  # seconds

def get_cached_dashboard():
    now = time.time()
    if cache["data"] and (now - cache["timestamp"]) < CACHE_TTL:
        return cache["data"]
    aggregated = fetch_all(API_URLS)
    result = build_dashboard(aggregated)
    cache["data"] = result
    cache["timestamp"] = now
    return result

print("Caching added (30s TTL).")

## Step 6: Complete Application

Full code with all components.

In [ ]:
import requests
import time
from flask import Flask, jsonify

app = Flask(__name__)

API_URLS = [
    "https://httpbin.org/json",
    "https://httpbin.org/ip",
    "https://httpbin.org/uuid"
]

def fetch_json(url, timeout=5):
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except requests.RequestException:
        return None

def fetch_all(urls):
    return {f"source_{i}": fetch_json(url) or {"error": "fetch_failed"}
            for i, url in enumerate(urls)}

def build_dashboard(aggregated):
    data = {k: v for k, v in aggregated.items() if v and "error" not in v}
    errors = [k for k, v in aggregated.items() if not v or "error" in v]
    return {"sources": len(aggregated), "data": data, "errors": errors}

cache = {"data": None, "timestamp": 0}

@app.route("/api/dashboard")
def dashboard():
    if cache["data"] and (time.time() - cache["timestamp"]) < 30:
        return jsonify(cache["data"])
    agg = fetch_all(API_URLS)
    result = build_dashboard(agg)
    cache["data"], cache["timestamp"] = result, time.time()
    return jsonify(result)

@app.route("/")
def home():
    return "<h1>API Aggregator</h1><p><a href='/api/dashboard'>Dashboard</a></p>"

if __name__ == "__main__":
    app.run(port=5000)

## Step 7: Test the Dashboard

Run the app and test with requests or curl.

In [ ]:
# Test without running server - call the logic directly
agg = fetch_all(API_URLS)
result = build_dashboard(agg)
print("Dashboard result:")
import json
print(json.dumps(result, indent=2)[:500])

## Extensions to Try

- Use `concurrent.futures` for parallel fetching
- Add more APIs (e.g., weather, quotes)
- Add query param to filter sources
- Add authentication for the dashboard endpoint
- Use Redis for caching instead of in-memory

## Key Takeaways

- Aggregate multiple APIs into one response
- Handle fetch errors gracefully
- Cache to reduce API load
- Flask routes can call helper functions

## 🎉 Week 13 Complete!

You've learned HTTP, requests, BeautifulSoup, REST, and Flask basics. Next week: Flask templates, forms, and SQLAlchemy!